In [ ]:
from modeldata import KLDTWLoss, GazeLLNArch
import numpy as np
import pandas as pd
import cv2 as cv

import torch
import torch.nn as nn
import torchvision.models as models
import pytorch_lightning as pl

from ncps.torch import CfCCell
from fractions import Fraction

import os
import random
from tqdm import tqdm
from typing import Tuple, List, Dict, Optional

import numpy as np
import scipy.io as sio
from PIL import Image
from scipy.ndimage import gaussian_filter

from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms    

ModuleNotFoundError: No module named 'modeldata'

In [ ]:
def _gaussian_heatmap(
    x: float,
    y: float,
    img_w: int,
    img_h: int,
    hmap_w: int,
    hmap_h: int,
    sigma: float = 1.5,
) -> np.ndarray:
    """
    Place a Gaussian blob at pixel (x, y) on a downsampled heatmap grid.

    Args:
        x, y      : fixation location in *original* image pixel coordinates
        img_w/h   : original image size (before downsampling)
        hmap_w/h  : heatmap size (= img / downsample)
        sigma     : Gaussian std-dev in heatmap pixels

    Returns:
        hmap : (hmap_h, hmap_w) float32 array summing to 1
    """
    hmap = np.zeros((hmap_h, hmap_w), dtype=np.float32)

    # scale fixation coords to heatmap space
    xi = int(round(x * hmap_w / img_w))
    yi = int(round(y * hmap_h / img_h))

    xi = np.clip(xi, 0, hmap_w - 1)
    yi = np.clip(yi, 0, hmap_h - 1)

    hmap[yi, xi] = 1.0
    hmap = gaussian_filter(hmap, sigma=sigma)

    total = hmap.sum()
    if total > 0:
        hmap /= total   

    return hmap

In [ ]:
def _pad_or_truncate(
    seq: List,
    max_len: int,
    pad_value,
) -> Tuple[List, List[bool]]:
    """Truncate to max_len and return (padded_seq, mask).
    mask[i] = True  →  real fixation
    mask[i] = False →  padding (loss should ignore these)
    """
    real_len = min(len(seq), max_len)
    mask = [True] * real_len + [False] * (max_len - real_len)
    seq = seq[:real_len] + [pad_value] * (max_len - real_len)
    return seq, mask

In [ ]:
def _center_gaussian(
    batch_size: int,
    hmap_h: int,
    hmap_w: int,
    device,
    sigma: float = 1.5,
) -> torch.Tensor:
    """
    Gaussian blob at image center using the existing _gaussian_heatmap utility.
    Shape: (B, 1, hmap_h, hmap_w)
    """
    hmap = _gaussian_heatmap(
        x=hmap_w / 2,                   # center x in heatmap coords — pass as if img_w == hmap_w
        y=hmap_h / 2,                   # center y
        img_w=hmap_w,
        img_h=hmap_h,
        hmap_w=hmap_w,
        hmap_h=hmap_h,
        sigma=sigma,
    )
    hmap_t = torch.from_numpy(hmap).to(device)          # (hmap_h, hmap_w)

    return hmap_t.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, -1, -1).clone()

In [ ]:
def extract_scanpaths(
    entry: Dict,
    min_len: int = 4,
    max_len: int = 8,
) -> List[Dict]:
    """
    Extract per-subject scanpaths from one OSIE entry.

    Returns a list of dicts, each with keys:
        img_name  : str
        fix_x     : list[float]  (length <= max_len)
        fix_y     : list[float]
        fix_dt    : list[float]  (durations in ms)
        mask      : list[bool]   (True = real, False = padding)
    """
    scanpaths = []
    img_name = entry['img']

    for subj in entry['subjects']:
        xs  = np.atleast_1d(np.array(subj['fix_x'],        dtype=np.float32)).tolist()
        ys  = np.atleast_1d(np.array(subj['fix_y'],        dtype=np.float32)).tolist()
        dts = np.atleast_1d(np.array(subj['fix_duration'], dtype=np.float32)).tolist()

        if len(xs) < min_len:
            continue  # discard short scanpaths per GazeLNN / tSPM-Net

        xs, mask = _pad_or_truncate(xs, max_len, pad_value=xs[-1])
        ys, _    = _pad_or_truncate(ys, max_len, pad_value=ys[-1])
        dts, _   = _pad_or_truncate(dts, max_len, pad_value=0.0)

        scanpaths.append({
            'img_name': img_name,
            'fix_x':    xs,
            'fix_y':    ys,
            'fix_dt':   dts,
            'mask':     mask,
        })

    return scanpaths

In [ ]:
class MITDataset(Dataset):
    """

    Each item is one (image, scanpath) pair — one subject's viewing of one
    image.  With 700 images × ~15 subjects the dataset has ~10 500 items
    before filtering.

    Args:
        data_root   : path to the osie/ folder containing stimuli/ and eye/
        split       : 'train' | 'val' | 'test'
        img_size    : (H, W) to resize images to. GazeLNN uses (256, 384).
        downsample  : spatial downsampling factor for heatmaps.
                      heatmap size = (img_size[0]//ds, img_size[1]//ds)
        min_len     : discard scanpaths shorter than this
        max_len     : truncate/pad scanpaths to this length
        sigma       : Gaussian std-dev (in heatmap pixels) for fixation blobs
        seed        : random seed for the 80/10/10 image split
    """

    def __init__(
        self,
        data_root: str,
        img_size: Tuple[int, int] = (256, 384),
        downsample: int = 8,
        min_len: int = 4,
        max_len: int = 8,
        sigma: float = 1.5,
        seed: int = 42,
    ):
        
        self.stimuli_dir = os.path.join(data_root, 'images')
        self.img_size    = img_size          # (H, W)
        self.hmap_size   = (img_size[0] // downsample, img_size[1] // downsample)
        self.max_len     = max_len
        self.sigma       = sigma

        # ---- load fixations ------------------------------------------------
        mat_path = os.path.join(data_root, 'eye', 'fixations.mat')
        raw = sio.loadmat(mat_path, simplify_cells=True)
        all_entries = raw['fixations']          # length 700

        # ---- deterministic image-level split --------------------------------
        n_total = len(all_entries)
        indices = list(range(n_total))

        chosen = indices        

        # ---- flatten to (image, subject) scanpath pairs --------------------
        self.samples: List[Dict] = []
        for idx in chosen:
            entry = all_entries[idx]
            scanpaths = extract_scanpaths(entry, min_len=min_len, max_len=max_len)
            self.samples.extend(scanpaths)

        # ---- image transforms ----------------------------------------------
        self.transform = transforms.Compose([
            transforms.Resize(img_size),          # PIL Resize takes (H, W)
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],       # ImageNet stats
                std =[0.229, 0.224, 0.225],
            ),
        ])

        print(
            f"[MITDataset]| "
            f"images={len(chosen)} | scanpath pairs={len(self.samples)}"
        )

    
    def __len__(self) -> int:
        return len(self.samples)

    
    def __getitem__(self, idx: int):
        sample   = self.samples[idx]
        img_H, img_W = self.img_size
        hmap_H, hmap_W = self.hmap_size

        #  load & transform image 
        img_path = os.path.join(self.stimuli_dir, sample['img_name'])
        img = Image.open(img_path).convert('RGB')

        # original size needed to scale fixation coords
        orig_W, orig_H = img.size   # PIL gives (W, H)

        img_tensor = self.transform(img)   # (3, H, W)

        # ---- build heatmap sequence ----------------------------------------
        heatmap_seq = []
        for i in range(self.max_len):
            if sample['mask'][i]:
                hmap = _gaussian_heatmap(
                    x=sample['fix_x'][i],
                    y=sample['fix_y'][i],
                    img_w=orig_W,
                    img_h=orig_H,
                    hmap_w=hmap_W,
                    hmap_h=hmap_H,
                    sigma=self.sigma,
                )
            else:
                # padding step — zero heatmap
                hmap = np.zeros((hmap_H, hmap_W), dtype=np.float32)

            heatmap_seq.append(hmap)

        heatmap_seq = torch.from_numpy(
            np.stack(heatmap_seq, axis=0)
        )  # (T, hmap_H, hmap_W)

        # ---- fixation durations (∆t for CfC) --------------------------------
        dt_seq = torch.tensor(sample['fix_dt'], dtype=torch.float32)  # (T,)

        # first fixation ∆t = 0 per the paper
        dt_seq[0] = 0.0

        # ---- padding mask ---------------------------------------------------
        mask = torch.tensor(sample['mask'], dtype=torch.bool)  # (T,)

        return img_tensor, heatmap_seq, dt_seq, mask

In [ ]:
def build_dataloaders(
    data_root: str,
    img_size: Tuple[int, int] = (256, 384),
    downsample: int = 8,
    min_len: int = 4,
    max_len: int = 8,
    sigma: float = 1.5,
    batch_size: int = 8,
    num_workers: int = 4,
    seed: int = 42,
) -> Tuple[DataLoader]:
    """


    Each batch yields:
        imgs         : (B, 3, H, W)       — normalised image tensor
        heatmap_seq  : (B, T, Hd, Wd)     — per-fixation Gaussian heatmaps
        dt_seq       : (B, T)             — fixation durations in ms
        padding_mask : (B, T)             — True where fixation is real
    """
    kwargs = dict(
        data_root  = data_root,
        img_size   = img_size,
        downsample = downsample,
        min_len    = min_len,
        max_len    = max_len,
        sigma      = sigma,
        seed       = seed,
    )


    test_ds  = MITDataset(split='test',  **kwargs)

    loader_kwargs = dict(
        batch_size  = batch_size,
        num_workers = num_workers,
        pin_memory  = True,
    )

    test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kwargs)

    return test_loader

In [ ]:
def eval_epoch(model, loader, criterion, device):
    """
    Runs one full pass over a validation or test dataloader.
    No gradient computation — inference only.

    Args:
        model     : GazeLLNArch instance
        loader    : val or test DataLoader
        criterion : KLDTWLoss instance
        device    : "cuda" or "cpu"

    Returns:
        avg_loss : float — mean loss over all batches
    """
    
    model.eval()
    with torch.no_grad():
        total_loss = 0.0

        for imgs, hmap_seq, dt_seq, mask in tqdm(loader, desc="Validating", leave=False):

            imgs     = imgs.to(device)
            hmap_seq = hmap_seq.to(device)
            mask     = mask.to(device)
            dt_seq = dt_seq.to(device)

            B      = imgs.shape[0]
            T      = hmap_seq.shape[1]
            hmap_H = hmap_seq.shape[2]
            hmap_W = hmap_seq.shape[3]

            with torch.no_grad():
                vis_features = model.extract_features(imgs)  # (B, 960)

            # prev_hmap = torch.zeros(B, 1, hmap_H, hmap_W, device=device)
            prev_hmap = _center_gaussian(B, hmap_H, hmap_W, device=device, sigma = 1.5)
            hx        = None
            # ts      = torch.ones(B, device=device)

            predictions = []

            for t in range(T):
                ts = dt_seq[:, t].view(-1, 1)
                out_hmap, hx = model(vis_features, prev_hmap, hx, ts)
                predictions.append(out_hmap.squeeze(1))
                prev_hmap = out_hmap

            predictions = torch.stack(predictions, dim=1)
            loss = criterion(predictions, hmap_seq, mask)
            total_loss += loss.item()

        return total_loss / len(loader)

In [ ]:
def evaluate_test(model, test_loader, device="cuda", checkpoint="best_model.pt"):
    """
    Loads the best saved weights and evaluates on the test set.

    Returns:
        test_loss : float
    """
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model = model.to(device)
    criterion = KLDTWLoss()

    test_loss = eval_epoch(model, test_loader, criterion, device)
    print(f"Test loss: {test_loss:.4f}")
    return test_loss